In [2]:
import win32com.client
import csv
from pathlib import Path
import pandas as pd
from datetime import datetime
import numpy as np

In [ ]:
# Выгрузка из 1С:
import win32com.client
import csv
from pathlib import Path
from datetime import datetime, timedelta
from typing import Optional, Dict, Any
import pandas as pd

# 0. Общие настройки

BASE_DIR = str(Path('1c_files').resolve())
OUT_DIR  = Path('export_from_1c')

# Заменить на реальный логин и пароль перед запуском
CONN_STRING = f'File="{BASE_DIR}";Usr="***";Pwd="***"'

DATE_FROM = datetime(2017, 4, 10, 0, 0, 0)
DATE_TO   = datetime(2025, 10, 29, 23, 59, 59)

PRICE_SLICE = datetime(2025, 10, 29, 23, 59, 59)

RETAIL_PRICE_NAME   = "Розничная30%"
PURCHASE_PRICE_NAME = "Закупочная"
RETAIL_COL = RETAIL_PRICE_NAME

PRICES_DATE_FROM = datetime(DATE_FROM.year - 10, 1, 1, 0, 0, 0)

OUT_DIR.mkdir(exist_ok=True)

v8 = win32com.client.Dispatch("V83.COMConnector")
conn = v8.Connect(CONN_STRING)

def export_query_to_csv(connection, query_text: str, csv_path: Path, params: Optional[Dict[str, Any]] = None) -> Path:
    """Выполнить запрос 1С и сохранить результат в CSV с разделителем ';'. Поддерживает параметры запроса."""
    q = connection.NewObject("Query", query_text)
    if params:
        for k, v in params.items():
            q.SetParameter(k, v)
    table = q.Execute().Unload()

    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f, delimiter=';')
        cols = [col.Name for col in table.Columns]
        writer.writerow(cols)
        for row in table:
            writer.writerow([getattr(row, col) for col in cols])

    return csv_path

def sort_for_asof(df: pd.DataFrame, on: str, by: Optional[str] = None) -> pd.DataFrame:
    cols = [on] + ([by] if by else [])
    return df.sort_values(cols).reset_index(drop=True)


def to_num_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.replace('\xa0', '', regex=False)
    s = s.str.replace(' ', '', regex=False)
    s = s.str.replace(',', '.', regex=False)
    return pd.to_numeric(s, errors='coerce')

# 1. Продажи по неделям

QUERY_SALES_WEEKS = '''
ВЫБРАТЬ
    НАЧАЛОПЕРИОДА(Прод.Период, НЕДЕЛЯ)       КАК НеделяНачало,
    ПРЕДСТАВЛЕНИЕ(Прод.Склад)               КАК Склад,
    Ном.Код                                 КАК КодТовара,
    СУММА(Прод.Количество)                  КАК Количество
ИЗ
    РегистрНакопления.Продажи КАК Прод
    ЛЕВОЕ СОЕДИНЕНИЕ
        Справочник.Номенклатура КАК Ном
    ПО Прод.Номенклатура = Ном.Ссылка
ГДЕ
    Прод.Период МЕЖДУ &D1 И &D2
СГРУППИРОВАТЬ ПО
    НАЧАЛОПЕРИОДА(Прод.Период, НЕДЕЛЯ),
    Прод.Склад,
    Ном.Код
'''

CSV_SALES = OUT_DIR / 'sales_weeks_by_code.csv'
export_query_to_csv(conn, QUERY_SALES_WEEKS, CSV_SALES, {"D1": DATE_FROM, "D2": DATE_TO})

# 2. Движения (Приход) для ДатыПоследнегоПоступления

QUERY_RECEIPTS = '''
ВЫБРАТЬ
    Движения.Период                         КАК Период,
    ПРЕДСТАВЛЕНИЕ(Движения.Склад)           КАК Склад,
    Ном.Код                                 КАК КодТовара
ИЗ
    РегистрНакопления.ТоварыНаСкладах КАК Движения
    ЛЕВОЕ СОЕДИНЕНИЕ
        Справочник.Номенклатура КАК Ном
    ПО Движения.Номенклатура = Ном.Ссылка
ГДЕ
    Движения.Период МЕЖДУ &D1 И &D2
    И Движения.ВидДвижения = ЗНАЧЕНИЕ(ВидДвиженияНакопления.Приход)
'''

CSV_MOVES = OUT_DIR / 'receipts_moves_by_code.csv'
export_query_to_csv(conn, QUERY_RECEIPTS, CSV_MOVES, {"D1": DATE_FROM, "D2": DATE_TO})

# 3. История цен

QUERY_PRICES_HISTORY = f'''
ВЫБРАТЬ
    Цены.Период                               КАК Период,
    Ном.Код                                   КАК КодТовара,
    Цены.ВидЦены.Наименование                  КАК ВидЦены,
    ПРЕДСТАВЛЕНИЕ(Цены.Характеристика)         КАК Характеристика,
    Цены.Цена                                 КАК Цена
ИЗ
    РегистрСведений.ЦеныНоменклатуры КАК Цены
    ЛЕВОЕ СОЕДИНЕНИЕ
        Справочник.Номенклатура КАК Ном
    ПО Цены.Номенклатура = Ном.Ссылка
ГДЕ
    Цены.Период МЕЖДУ &D1 И &D2
    И Цены.ВидЦены.Наименование В ("{RETAIL_PRICE_NAME}", "{PURCHASE_PRICE_NAME}")
'''

CSV_PRICES = OUT_DIR / 'prices_history.csv'
export_query_to_csv(conn, QUERY_PRICES_HISTORY, CSV_PRICES, {"D1": PRICES_DATE_FROM, "D2": DATE_TO})

# 4. Список всех недель (понедельник 00:00:00)

start_date_only = datetime(DATE_FROM.year, DATE_FROM.month, DATE_FROM.day)
start_monday = datetime(DATE_FROM.year, DATE_FROM.month, DATE_FROM.day, 0, 0, 0) - timedelta(days=start_date_only.weekday())
end_day = datetime(DATE_TO.year, DATE_TO.month, DATE_TO.day, 0, 0, 0)

week_starts = []
cur = start_monday
while cur <= end_day:
    week_starts.append(cur)  # naive для 1С
    cur += timedelta(days=7)

# 5. (остатки на начало недели ∪ продажи недели) + атрибуты + срезы цен

QUERY_STOCK_DIM = f'''
ВЫБРАТЬ
    Ном.Код                                   КАК КодТовара,
    ПРЕДСТАВЛЕНИЕ(Ключи.Склад)                КАК Склад,

    ПРЕДСТАВЛЕНИЕ(Ключи.Номенклатура)         КАК Номенклатура,
    ПРЕДСТАВЛЕНИЕ(Ном.Родитель.Родитель)      КАК Папка1,
    ПРЕДСТАВЛЕНИЕ(Ном.Родитель)               КАК Папка2,
    ПРЕДСТАВЛЕНИЕ(Ном.ЕдиницаИзмерения)       КАК ЕдиницаИзмерения,
    ПРЕДСТАВЛЕНИЕ(Ном.ТоварнаяКатегория)      КАК ТоварнаяКатегория,
    ПРЕДСТАВЛЕНИЕ(Поставщики.Поставщик)       КАК Поставщик,

    ЗакупкаСрез.ЗакупочнаяЦена                КАК ЗакупочнаяЦенаСрез,
    РозницаСрез.РозничнаяЦена                 КАК РозничнаяЦенаСрез,

    ЕСТЬNULL(Ост.КоличествоОстаток, 0)        КАК ОстатокНачалоНедели

ИЗ
(
    // ключи из остатков на начало недели
    ВЫБРАТЬ РАЗЛИЧНЫЕ
        Ост0.Склад        КАК Склад,
        Ост0.Номенклатура КАК Номенклатура
    ИЗ
        РегистрНакопления.ТоварыНаСкладах.Остатки(&Момент) КАК Ост0

    ОБЪЕДИНИТЬ

    // ключи из продаж за неделю (чтобы включить товары с остатком 0)
    ВЫБРАТЬ РАЗЛИЧНЫЕ
        Прод.Склад        КАК Склад,
        Прод.Номенклатура КАК Номенклатура
    ИЗ
        РегистрНакопления.Продажи КАК Прод
    ГДЕ
        Прод.Период МЕЖДУ &Момент И &НеделяКонец
) КАК Ключи

ЛЕВОЕ СОЕДИНЕНИЕ
    РегистрНакопления.ТоварыНаСкладах.Остатки(&Момент) КАК Ост
ПО
    Ост.Склад = Ключи.Склад
    И Ост.Номенклатура = Ключи.Номенклатура

ЛЕВОЕ СОЕДИНЕНИЕ
    Справочник.Номенклатура КАК Ном
ПО
    Ключи.Номенклатура = Ном.Ссылка

ЛЕВОЕ СОЕДИНЕНИЕ
(
    ВЫБРАТЬ
        НомПост.Номенклатура                 КАК Номенклатура,
        МАКСИМУМ(НомПост.Поставщик)         КАК Поставщик
    ИЗ
        РегистрСведений.НоменклатураПоставщиков КАК НомПост
    СГРУППИРОВАТЬ ПО
        НомПост.Номенклатура
) КАК Поставщики
ПО
    Ключи.Номенклатура = Поставщики.Номенклатура

ЛЕВОЕ СОЕДИНЕНИЕ
(
    ВЫБРАТЬ
        Цены.Номенклатура                   КАК Номенклатура,
        Цены.Цена                           КАК ЗакупочнаяЦена
    ИЗ
        РегистрСведений.ЦеныНоменклатуры.СрезПоследних(
            ДАТАВРЕМЯ({PRICE_SLICE.year},{PRICE_SLICE.month},{PRICE_SLICE.day},{PRICE_SLICE.hour},{PRICE_SLICE.minute},{PRICE_SLICE.second})
        ) КАК Цены
    ГДЕ
        Цены.ВидЦены.Наименование = "{PURCHASE_PRICE_NAME}"
) КАК ЗакупкаСрез
ПО
    Ключи.Номенклатура = ЗакупкаСрез.Номенклатура

ЛЕВОЕ СОЕДИНЕНИЕ
(
    ВЫБРАТЬ
        Цены.Номенклатура                   КАК Номенклатура,
        Цены.Цена                           КАК РозничнаяЦена
    ИЗ
        РегистрСведений.ЦеныНоменклатуры.СрезПоследних(
            ДАТАВРЕМЯ({PRICE_SLICE.year},{PRICE_SLICE.month},{PRICE_SLICE.day},{PRICE_SLICE.hour},{PRICE_SLICE.minute},{PRICE_SLICE.second})
        ) КАК Цены
    ГДЕ
        Цены.ВидЦены.Наименование = "{RETAIL_PRICE_NAME}"
) КАК РозницаСрез
ПО
    Ключи.Номенклатура = РозницаСрез.Номенклатура
'''

stock_rows = []

for dt_py in week_starts:
    week_end = dt_py + timedelta(days=7) - timedelta(seconds=1)

    q = conn.NewObject("Query", QUERY_STOCK_DIM)
    q.SetParameter("Момент", dt_py)
    q.SetParameter("НеделяКонец", week_end)

    table = q.Execute().Unload()

    for row in table:
        stock_rows.append([
            dt_py,
            str(row.Склад),
            str(row.КодТовара),
            str(row.Номенклатура),
            str(row.Папка1),
            str(row.Папка2),
            str(row.ЕдиницаИзмерения),
            str(row.ТоварнаяКатегория),
            str(row.Поставщик),
            row.ЗакупочнаяЦенаСрез,
            row.РозничнаяЦенаСрез,
            row.ОстатокНачалоНедели
        ])

base_df = pd.DataFrame(
    stock_rows,
    columns=[
        'НеделяНачало',
        'Склад',
        'КодТовара',
        'Номенклатура',
        'Папка1',
        'Папка2',
        'ЕдиницаИзмерения',
        'ТоварнаяКатегория',
        'Поставщик',
        'ЗакупочнаяЦенаСрез',
        'РозничнаяЦенаСрез',
        'ОстатокНачалоНедели'
    ]
)

base_df['НеделяНачало'] = pd.to_datetime(base_df['НеделяНачало'], errors='coerce', utc=True)
base_df = base_df.dropna(subset=['НеделяНачало', 'Склад', 'КодТовара'])

base_df['ЗакупочнаяЦенаСрез'] = pd.to_numeric(base_df['ЗакупочнаяЦенаСрез'], errors='coerce')
base_df['РозничнаяЦенаСрез']  = pd.to_numeric(base_df['РозничнаяЦенаСрез'], errors='coerce')
base_df['ОстатокНачалоНедели'] = pd.to_numeric(base_df['ОстатокНачалоНедели'], errors='coerce').fillna(0)

# 6. Подмешиваем продажи

sales = pd.read_csv(
    CSV_SALES,
    sep=';',
    dtype={'КодТовара': str, 'Склад': str},
    low_memory=False
)
sales['НеделяНачало'] = pd.to_datetime(sales['НеделяНачало'], errors='coerce', utc=True)
sales['Количество'] = pd.to_numeric(sales['Количество'], errors='coerce')
sales = sales.dropna(subset=['НеделяНачало', 'Склад', 'КодТовара'])

df = base_df.merge(
    sales[['НеделяНачало', 'Склад', 'КодТовара', 'Количество']],
    on=['НеделяНачало', 'Склад', 'КодТовара'],
    how='left'
)
df['Количество'] = df['Количество'].fillna(0)

# 7. Цены по времени

prices = pd.read_csv(
    CSV_PRICES,
    sep=';',
    dtype={'КодТовара': str, 'ВидЦены': str, 'Характеристика': str, 'Цена': str},
    low_memory=False
)

prices['Период'] = pd.to_datetime(prices['Период'], errors='coerce', utc=True)
prices['Цена'] = to_num_series(prices['Цена'])
prices['Характеристика'] = prices['Характеристика'].fillna('')
prices = prices.dropna(subset=['Период', 'КодТовара', 'ВидЦены'])

prices_empty_char = prices[prices['Характеристика'] == ''].copy()
prices_use = prices_empty_char if not prices_empty_char.empty else prices

retail = prices_use[prices_use['ВидЦены'] == RETAIL_PRICE_NAME].copy()
retail = retail.rename(columns={'Цена': 'РозничнаяЦена'})
retail = retail.drop_duplicates(['КодТовара', 'Период'], keep='last')

purchase = prices_use[prices_use['ВидЦены'] == PURCHASE_PRICE_NAME].copy()
purchase = purchase.rename(columns={'Цена': 'ЗакупочнаяЦена'})
purchase = purchase.drop_duplicates(['КодТовара', 'Период'], keep='last')

df = sort_for_asof(df, on='НеделяНачало', by='КодТовара')
retail = sort_for_asof(retail[['Период', 'КодТовара', 'РозничнаяЦена']], on='Период', by='КодТовара')
purchase = sort_for_asof(purchase[['Период', 'КодТовара', 'ЗакупочнаяЦена']], on='Период', by='КодТовара')

df = pd.merge_asof(
    df,
    retail,
    left_on='НеделяНачало',
    right_on='Период',
    by='КодТовара',
    direction='backward',
    allow_exact_matches=True
).rename(columns={'РозничнаяЦена': 'РозничнаяЦена_back'}).drop(columns=['Период'], errors='ignore')

df[RETAIL_COL] = df['РозничнаяЦена_back']

df = sort_for_asof(df, on='НеделяНачало', by='КодТовара')
df = pd.merge_asof(
    df,
    purchase,
    left_on='НеделяНачало',
    right_on='Период',
    by='КодТовара',
    direction='backward',
    allow_exact_matches=True
).rename(columns={'ЗакупочнаяЦена': 'ЗакупочнаяЦена_back'}).drop(columns=['Период'], errors='ignore')

df['ЗакупочнаяЦена'] = df['ЗакупочнаяЦена_back']

df = df.drop(columns=['РозничнаяЦена_back', 'ЗакупочнаяЦена_back'], errors='ignore')

# 8. ДатаПоследнегоПоступления

moves = pd.read_csv(
    CSV_MOVES,
    sep=';',
    dtype={'КодТовара': str, 'Склад': str},
    low_memory=False
)
moves['Период'] = pd.to_datetime(moves['Период'], errors='coerce', utc=True)
moves = moves.dropna(subset=['Период', 'КодТовара', 'Склад'])

df = sort_for_asof(df, on='НеделяНачало', by=None)
moves = sort_for_asof(moves, on='Период', by=None)

df_list = []
for wh, df_wh in df.groupby('Склад', sort=False):
    mv_wh = moves[moves['Склад'] == wh].copy()
    if mv_wh.empty:
        df_wh['ДатаПоследнегоПоступления'] = pd.NaT
        df_list.append(df_wh)
        continue

    df_wh = sort_for_asof(df_wh, on='НеделяНачало', by='КодТовара')
    mv_wh = sort_for_asof(mv_wh, on='Период', by='КодТовара').rename(columns={'Период': 'ДатаПоследнегоПоступления'})

    df_wh = pd.merge_asof(
        df_wh,
        mv_wh[['ДатаПоследнегоПоступления', 'КодТовара']],
        left_on='НеделяНачало',
        right_on='ДатаПоследнегоПоступления',
        by='КодТовара',
        direction='backward',
        allow_exact_matches=True
    )
    df_list.append(df_wh)

df = pd.concat(df_list, ignore_index=True)

# 9. Финальные колонки и сохранение

cols_order = [
    'НеделяНачало',
    'Склад',
    'КодТовара',
    'Номенклатура',
    'Папка1',
    'Папка2',
    'Количество',
    RETAIL_COL,
    'ЕдиницаИзмерения',
    'ТоварнаяКатегория',
    'Поставщик',
    'ЗакупочнаяЦена',
    'ДатаПоследнегоПоступления',
    'ОстатокНачалоНедели'
]

df = df.reindex(columns=cols_order)

FINAL_PATH = OUT_DIR / 'full_features_no_future_prices.csv'
df.to_csv(FINAL_PATH, sep=';', index=False, encoding='utf-8')

print('Готовый файл:', FINAL_PATH)
print('Строк:', len(df))
print('Пустых Розничных30% (NaN):', int(df[RETAIL_COL].isna().sum()))
print('Пустых Закупочная (NaN):', int(df['ЗакупочнаяЦена'].isna().sum()))
print('Строк с ОстатокНачалоНедели=0 и Продажи>0:', int(((df['ОстатокНачалоНедели'] == 0) & (df['Количество'] > 0)).sum()))


In [316]:
# Как выглядят изначальные данные
df = pd.read_csv(r"export_from_1c\full_features.csv", sep=';')
df

,НеделяНачало,Склад,КодТовара,Номенклатура,Папка1,Папка2,Количество,Розничная30%,ЕдиницаИзмерения,ТоварнаяКатегория,Поставщик,ЗакупочнаяЦена,ДатаПоследнегоПоступления,ОстатокНачалоНедели
0,2017-04-10 00:00:00+00:00,Магазин,00-00000030,Биогумус Флорист Бутон 120мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,62.0,шт,NaN,NaN,47.34,NaN,10.0
1,2017-04-10 00:00:00+00:00,Магазин,00-00000031,Биогумус Флорист Микро 120мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,62.0,шт,NaN,NaN,47.34,NaN,9.0
2,2017-04-10 00:00:00+00:00,Магазин,00-00000032,Биогумус Флорист Рост 120мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",1.0,62.0,шт,NaN,NaN,47.34,NaN,8.0
3,2017-04-10 00:00:00+00:00,Магазин,00-00000033,БиоМастер для орхидей. Органоминеральное коспл...,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,62.0,шт,NaN,NaN,47.15,NaN,2.0
4,2017-04-10 00:00:00+00:00,Магазин,00-00000034,Бона Форте для орхидей 285мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",1.0,110.0,шт,NaN,"ИП Алексеев Н.С, ПЕНЗА САДОВИТА",84.24,NaN,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2760046,2025-10-27 00:00:00+00:00,Новый Магазин,РТ-00037957,"Горшок Афина 2,2л с поддоном",09 Оборудование для выращивания растений,"05 Горшки, кашпо, держатели, колышки",0.0,95.0,шт,NaN,Колхоз,64.06,2025-10-26 19:00:48+00:00,3.0
2760047,2025-10-27 00:00:00+00:00,Новый Магазин,РТ-00037960,"Горшок Виола 1,0л.с поддоном",09 Оборудование для выращивания растений,"05 Горшки, кашпо, держатели, колышки",3.0,60.0,шт,NaN,Колхоз,39.49,2025-10-26 19:00:48+00:00,1.0
2760048,2025-10-27 00:00:00+00:00,Новый Магазин,РТ-00037969,Веник сорго 6-и лучевой Узбекистан,18 Товары для быта,02 Товары для дома,1.0,210.0,шт,NaN,Колхоз,141.71,2025-10-22 19:05:05+00:00,0.0
2760049,2025-10-27 00:00:00+00:00,Новый Магазин,РТ-00038009,Мыло жидкое ВИТАНО 500мл 75708,18 Товары для быта,03 Банные принадлежности,1.0,84.0,шт,NaN,Колхоз,63.51,NaN,0.0


In [ ]:
# Преобразуем 'НеделяНачало' и 'ДатаПоследнегоПоступления' в datetime
df = df.copy()
df["НеделяНачало"] = (
    pd.to_datetime(df["НеделяНачало"], utc=True)
    .dt.tz_convert(None)
)
df["ДатаПоследнегоПоступления"] = (
    pd.to_datetime(df["ДатаПоследнегоПоступления"], utc=True)
    .dt.tz_convert(None)
)

In [319]:
# Удалим все строки где пропущено значение Розничная30%, ведь такие товарны не могли продаваться
df = df.dropna(subset=['Розничная30%'])

In [320]:
# Удалим все строки где пропущено значение ЗакупочнаяЦена
df = df.dropna(subset=['ЗакупочнаяЦена'])

In [321]:
# Строки, где в одной и той же неделе один и тот же КодТовара встречается больше 1 раза
dups = (
    df[df.duplicated(subset=["НеделяНачало", "КодТовара", 'Склад'], keep=False)]
    .sort_values(["НеделяНачало", "КодТовара", 'Склад'])
)

dups
# Вывод, дубликаты только у ГалошиПВХ из-за двух размеров

,НеделяНачало,Склад,КодТовара,Номенклатура,Папка1,Папка2,Количество,Розничная30%,ЕдиницаИзмерения,ТоварнаяКатегория,Поставщик,ЗакупочнаяЦена,ДатаПоследнегоПоступления,ОстатокНачалоНедели


In [322]:
# Проверим сколько пропусков в каждой строке
df.isna().sum()

НеделяНачало                       0
Склад                              2
КодТовара                          0
Номенклатура                       0
Папка1                          4700
Папка2                             0
Количество                         0
Розничная30%                       0
ЕдиницаИзмерения                   0
ТоварнаяКатегория            2730347
Поставщик                     294163
ЗакупочнаяЦена                     0
ДатаПоследнегоПоступления      85169
ОстатокНачалоНедели                0
dtype: int64

In [323]:
# Удалим значения без склада
df = df.dropna(subset=['Склад'])

In [324]:
# Какие категории "ЕдиницаИзмерения" вообще существуют?	
df['ЕдиницаИзмерения'].value_counts()

ЕдиницаИзмерения
шт    2730274
м        6599
Name: count, dtype: int64

In [325]:
# Заполним пропуски в 'ТоварнаяКатегория'
df = df.copy()
df['ТоварнаяКатегория'] = df['ТоварнаяКатегория'].fillna('Штучный товар')

In [327]:
# Заполним пропуски в 'Поставщик'
df['Поставщик'] = df['Поставщик'].fillna('Неизвестный поставщик')

In [328]:
# Заполним пропуски в 'ЗакупочнаяЦена'
# Пропущена цена только у Насос вибрационный НВТ-210/10, заменим на 833 (прим стоимость)
df.loc[df['ЗакупочнаяЦена'].isna(), 'ЗакупочнаяЦена'] = df.loc[df['ЗакупочнаяЦена'].isna(), 'Розничная30%'] * 0.7


In [329]:
# Удалим строки с Насос вибрационный НВТ-210/10, он был бракованный
df = df[df['Номенклатура'] != 'Насос вибрационный НВТ-210/10']

In [ ]:
# Заполним пропуски в Папка1
df["Папка1"] = df["Папка1"].fillna("разобрать")

In [331]:
# Удалим колонку Склад
df = df.drop(columns=['Склад'])

In [332]:
# Проверим сколько пропусков в каждой строке
df.isna().sum()

НеделяНачало                     0
КодТовара                        0
Номенклатура                     0
Папка1                           0
Папка2                           0
Количество                       0
Розничная30%                     0
ЕдиницаИзмерения                 0
ТоварнаяКатегория                0
Поставщик                        0
ЗакупочнаяЦена                   0
ДатаПоследнегоПоступления    85167
ОстатокНачалоНедели              0
dtype: int64

In [333]:
df

,НеделяНачало,КодТовара,Номенклатура,Папка1,Папка2,Количество,Розничная30%,ЕдиницаИзмерения,ТоварнаяКатегория,Поставщик,ЗакупочнаяЦена,ДатаПоследнегоПоступления,ОстатокНачалоНедели
0,2017-04-10,00-00000030,Биогумус Флорист Бутон 120мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,62.0,шт,Штучный товар,Неизвестный поставщик,47.34,NaT,10.0
1,2017-04-10,00-00000031,Биогумус Флорист Микро 120мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,62.0,шт,Штучный товар,Неизвестный поставщик,47.34,NaT,9.0
2,2017-04-10,00-00000032,Биогумус Флорист Рост 120мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",1.0,62.0,шт,Штучный товар,Неизвестный поставщик,47.34,NaT,8.0
3,2017-04-10,00-00000033,БиоМастер для орхидей. Органоминеральное коспл...,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,62.0,шт,Штучный товар,Неизвестный поставщик,47.15,NaT,2.0
4,2017-04-10,00-00000034,Бона Форте для орхидей 285мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",1.0,110.0,шт,Штучный товар,"ИП Алексеев Н.С, ПЕНЗА САДОВИТА",84.24,NaT,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2760046,2025-10-27,РТ-00037957,"Горшок Афина 2,2л с поддоном",09 Оборудование для выращивания растений,"05 Горшки, кашпо, держатели, колышки",0.0,95.0,шт,Штучный товар,Колхоз,64.06,2025-10-26 19:00:48,3.0
2760047,2025-10-27,РТ-00037960,"Горшок Виола 1,0л.с поддоном",09 Оборудование для выращивания растений,"05 Горшки, кашпо, держатели, колышки",3.0,60.0,шт,Штучный товар,Колхоз,39.49,2025-10-26 19:00:48,1.0
2760048,2025-10-27,РТ-00037969,Веник сорго 6-и лучевой Узбекистан,18 Товары для быта,02 Товары для дома,1.0,210.0,шт,Штучный товар,Колхоз,141.71,2025-10-22 19:05:05,0.0
2760049,2025-10-27,РТ-00038009,Мыло жидкое ВИТАНО 500мл 75708,18 Товары для быта,03 Банные принадлежности,1.0,84.0,шт,Штучный товар,Колхоз,63.51,NaT,0.0


In [337]:
# Сложим значения двух магазинов

key_cols = ['НеделяНачало', 'КодТовара']

def choose_unit(s):
    vals = s.dropna().astype(str)
    if vals.empty:
        return np.nan
    mask_m = vals.str.contains('м', case=False)
    return vals[mask_m].iloc[0] if mask_m.any() else vals.iloc[0]

def choose_category(s):
    priority = {
        'Укрывной материал (отрезной)': 3,
        'Укрывной материал': 2,
        'Штучный товар': 1,
    }
    vals = s.dropna().astype(str)
    if vals.empty:
        return np.nan
    scores = vals.map(lambda x: priority.get(x, 0))
    return vals.iloc[scores.argmax()]

def choose_supplier(s):
    vals = s.dropna().astype(str)
    if vals.empty:
        return np.nan
    good = vals[vals != 'Неизвестный поставщик']
    return good.iloc[0] if len(good) else vals.iloc[0]

agg_df = (
    df
    .groupby(key_cols, as_index=False)
    .agg({
        'Номенклатура': 'first',
        'Папка1': 'first',
        'Папка2': 'first',
        'Количество': 'sum',
        'Розничная30%': 'mean',
        'ЕдиницаИзмерения': choose_unit,
        'ТоварнаяКатегория': choose_category,
        'Поставщик': choose_supplier,
        'ЗакупочнаяЦена': 'mean',
        'ДатаПоследнегоПоступления': 'max',
        'ОстатокНачалоНедели': 'sum',
    })
)


In [ ]:
# Удалим все записи, где Количество >= ОстатокНачалоНедели. Чтобы обучать модель на тех значениях,
# где точно удовлетворён спрос и нет искуственного ограничения
agg_df = agg_df[agg_df['Количество'] < agg_df['ОстатокНачалоНедели']]

In [352]:
agg_df

,НеделяНачало,КодТовара,Номенклатура,Папка1,Папка2,Количество,Розничная30%,ЕдиницаИзмерения,ТоварнаяКатегория,Поставщик,ЗакупочнаяЦена,ДатаПоследнегоПоступления,ОстатокНачалоНедели
0,2017-04-10,00-00000030,Биогумус Флорист Бутон 120мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,62.0,шт,Штучный товар,Неизвестный поставщик,47.34,NaT,10.0
1,2017-04-10,00-00000031,Биогумус Флорист Микро 120мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,62.0,шт,Штучный товар,Неизвестный поставщик,47.34,NaT,9.0
2,2017-04-10,00-00000032,Биогумус Флорист Рост 120мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",1.0,62.0,шт,Штучный товар,Неизвестный поставщик,47.34,NaT,8.0
3,2017-04-10,00-00000033,БиоМастер для орхидей. Органоминеральное коспл...,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,62.0,шт,Штучный товар,Неизвестный поставщик,47.15,NaT,2.0
4,2017-04-10,00-00000034,Бона Форте для орхидей 285мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",1.0,110.0,шт,Штучный товар,"ИП Алексеев Н.С, ПЕНЗА САДОВИТА",84.24,NaT,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2639927,2025-10-27,РТ-00038159,90х90х65 уголок крепежный ОЦ,03 Стройка,19 Скобяные изделия ШТУЧНЫЕ,0.0,46.0,шт,Штучный товар,Неизвестный поставщик,19.68,2025-10-25 20:00:00,59.0
2639928,2025-10-27,РТ-00038160,90х90х65 уголок крепежный усиленный ОЦ,03 Стройка,19 Скобяные изделия ШТУЧНЫЕ,0.0,36.0,шт,Штучный товар,Неизвестный поставщик,15.40,2025-10-25 20:00:00,60.0
2639929,2025-10-27,РТ-00038161,М10 анкер латунный забивной (цанга) 2шт.,03 Стройка,19 Скобяные изделия ШТУЧНЫЕ,0.0,57.0,шт,Штучный товар,Неизвестный поставщик,24.51,2025-10-25 20:00:00,9.0
2639930,2025-10-27,РТ-00038162,М12 16х50 анкер забивной,03 Стройка,19 Скобяные изделия ШТУЧНЫЕ,0.0,20.0,шт,Штучный товар,Неизвестный поставщик,8.35,2025-10-25 20:00:00,37.0


In [355]:
# Пропуски дат заполним 2017-04-02 12:00:00. Именно так в 1с
agg_df = agg_df.copy()
agg_df['ДатаПоследнегоПоступления'] = agg_df['ДатаПоследнегоПоступления'].fillna('2017-04-02 12:00:00')

In [356]:
# Приведём ДатаПоследнегоПоступления целиком к актуальному типу 
df["ДатаПоследнегоПоступления"] = (
    pd.to_datetime(df["ДатаПоследнегоПоступления"], utc=True)
        .dt.tz_convert(None)
)

In [357]:
# Проверим сколько пропусков в каждой строке
agg_df.isna().sum()

НеделяНачало                 0
КодТовара                    0
Номенклатура                 0
Папка1                       0
Папка2                       0
Количество                   0
Розничная30%                 0
ЕдиницаИзмерения             0
ТоварнаяКатегория            0
Поставщик                    0
ЗакупочнаяЦена               0
ДатаПоследнегоПоступления    0
ОстатокНачалоНедели          0
dtype: int64

In [358]:
# Возьмем все уникальные недели
df_weeks = df['НеделяНачало'].unique()

In [ ]:
# Создадим колонки temp_mean_week, precip_sum_week, temp_max_week, temp_min_week
import requests
import pandas as pd
import numpy as np

#  0. df_weeks: уникальные недели 

if isinstance(df_weeks, pd.Series):
    weeks = df_weeks
elif isinstance(df_weeks, pd.DataFrame):
    weeks = df_weeks.iloc[:, 0]
else:
    weeks = pd.Series(np.array(df_weeks).ravel())

weeks = pd.to_datetime(weeks)

try:
    weeks = weeks.dt.tz_convert(None)
except TypeError:
    pass

df_weeks = pd.DataFrame({"НеделяНачало": weeks})

start_date = weeks.min()
end_date = weeks.max() + pd.Timedelta(days=6)

#  1. Координаты 

LATITUDE = 53.221010
LONGITUDE = 50.634394

#  2. Запрос к Open-Meteo 

url = "https://archive-api.open-meteo.com/v1/archive"

all_daily = []

cur_start = start_date
while cur_start <= end_date:
    cur_end = min(cur_start + pd.DateOffset(years=1) - pd.Timedelta(days=1), end_date)

    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": cur_start.strftime("%Y-%m-%d"),
        "end_date": cur_end.strftime("%Y-%m-%d"),
        "daily": (
            "temperature_2m_mean,"
            "temperature_2m_max,"
            "temperature_2m_min,"
            "precipitation_sum"
        ),
        "timezone": "auto",
    }

    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()

    daily_part = pd.DataFrame({
        "date": pd.to_datetime(data["daily"]["time"]),
        "temp_mean": data["daily"]["temperature_2m_mean"],
        "temp_max": data["daily"]["temperature_2m_max"],
        "temp_min": data["daily"]["temperature_2m_min"],
        "precip": data["daily"]["precipitation_sum"],
    })

    all_daily.append(daily_part)

    cur_start = cur_end + pd.Timedelta(days=1)

daily = pd.concat(all_daily, ignore_index=True)

#  3. Агрегация по неделям

daily["НеделяНачало"] = daily["date"] - pd.to_timedelta(daily["date"].dt.weekday, unit="D")

weekly_weather = (
    daily
    .groupby("НеделяНачало", as_index=False)
    .agg(
        temp_mean_week=("temp_mean", "mean"),
        precip_sum_week=("precip", "sum"),
        temp_max_week=("temp_max", "max"),
        temp_min_week=("temp_min", "min"),
    )
)

# 4. Итоговый df

df_weeks_with_weather = df_weeks.merge(
    weekly_weather,
    on="НеделяНачало",
    how="left"
)


In [364]:
# Добавим данные о погоде к основному agg_df
weather_cols = ["temp_mean_week", "precip_sum_week", "temp_max_week", "temp_min_week"]

agg_df = agg_df.drop(columns=weather_cols, errors="ignore") \
               .merge(df_weeks_with_weather, on="НеделяНачало", how="left")


In [365]:
# Добавим столбец НДС
agg_df['НДС'] = agg_df['НеделяНачало'].apply(lambda d: 0.18 if d.year < 2019 else 0.20)

In [366]:
# Добавим колонку сколько праздничных дней в неделе
import holidays

years = agg_df["НеделяНачало"].dt.year.unique()
ru_holidays = holidays.Russia(years=years)

def count_days_off_window(start_date):
    days = pd.date_range(start=start_date, periods=7, freq="D")
    # выходной = только официальный праздник/перенос
    return sum(d.date() in ru_holidays for d in days)

agg_df["days_off_in_week"] = agg_df["НеделяНачало"].apply(count_days_off_window)


In [367]:
# Добавим лаг 2 недели по продажам
agg_df = agg_df.sort_values(["НеделяНачало"])

agg_df["sales_lag_2w"] = (
    agg_df
    .groupby("КодТовара")["Количество"]
    .shift(2)
)

In [368]:
agg_df

,НеделяНачало,КодТовара,Номенклатура,Папка1,Папка2,Количество,Розничная30%,ЕдиницаИзмерения,ТоварнаяКатегория,Поставщик,ЗакупочнаяЦена,ДатаПоследнегоПоступления,ОстатокНачалоНедели,temp_mean_week,precip_sum_week,temp_max_week,temp_min_week,НДС,days_off_in_week,sales_lag_2w
0,2017-04-10,00-00000030,Биогумус Флорист Бутон 120мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,62.0,шт,Штучный товар,Неизвестный поставщик,47.34,2017-04-02 12:00:00,10.0,6.142857,7.9,17.2,-5.3,0.18,0,NaN
1498,2017-04-10,00-00002263,Бархатцы Карнавальный дебют Гавриш,14 Семена,"06 Семена цветов, кустарников и деревьев",1.0,11.0,шт,Штучный товар,СемОпт ООО (ИП Бориско В.Н.)(Бориско Т.В.),7.12,2017-04-02 12:00:00,4.0,6.142857,7.9,17.2,-5.3,0.18,0,NaN
1497,2017-04-10,00-00002261,Барвинок Лилипут смесь Евро,14 Семена,"06 Семена цветов, кустарников и деревьев",0.0,10.5,шт,Штучный товар,Неизвестный поставщик,7.88,2017-04-02 12:00:00,3.0,6.142857,7.9,17.2,-5.3,0.18,0,NaN
1496,2017-04-10,00-00002260,Банан комнатный Аэлита,14 Семена,"06 Семена цветов, кустарников и деревьев",0.0,33.0,шт,Штучный товар,Неизвестный поставщик,22.06,2017-04-02 12:00:00,7.0,6.142857,7.9,17.2,-5.3,0.18,0,NaN
1495,2017-04-10,00-00002259,Бальзамин Экзотик Гавриш,14 Семена,"06 Семена цветов, кустарников и деревьев",1.0,11.0,шт,Штучный товар,"ИП Алексеев Н.С, ПЕНЗА САДОВИТА",7.12,2017-04-02 12:00:00,8.0,6.142857,7.9,17.2,-5.3,0.18,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2487656,2025-10-27,РТ-00019031,Репа Голден Болл Аэлита,14 Семена,04 Семена овощных культур,0.0,24.0,шт,Штучный товар,Колхоз,9.27,2024-09-16 12:11:55,31.0,6.257143,18.3,11.5,1.8,0.20,0,0.0
2487657,2025-10-27,РТ-00019032,Ромашка садовая Принцесса Аэлита,14 Семена,"06 Семена цветов, кустарников и деревьев",0.0,26.0,шт,Штучный товар,Колхоз,10.51,2024-06-24 13:43:59,7.0,6.257143,18.3,11.5,1.8,0.20,0,0.0
2487658,2025-10-27,РТ-00019033,Руккола (индау) Кореянка Аэлита,14 Семена,05 Семена пряных и лекарственных культур и трав,0.0,29.0,шт,Штучный товар,Колхоз,10.51,2025-09-13 10:10:05,61.0,6.257143,18.3,11.5,1.8,0.20,0,1.0
2487636,2025-10-27,РТ-00018998,Свекла Джулия Аэлита,14 Семена,04 Семена овощных культур,0.0,29.0,шт,Штучный товар,"ИП Алексеев Н.С, ПЕНЗА САДОВИТА",10.81,2025-08-29 11:27:58,80.0,6.257143,18.3,11.5,1.8,0.20,0,0.0


In [370]:
# Заполним пропуски в данных для колонки лаг 2 недели
median_lag2 = agg_df["sales_lag_2w"].median()
agg_df["sales_lag_2w"] = agg_df["sales_lag_2w"].fillna(median_lag2)

In [371]:
# Отсортируем по возрастанию по колонке НеделяНачало
agg_df = agg_df.sort_values('НеделяНачало', ascending=True)

In [372]:
agg_df

,НеделяНачало,КодТовара,Номенклатура,Папка1,Папка2,Количество,Розничная30%,ЕдиницаИзмерения,ТоварнаяКатегория,Поставщик,ЗакупочнаяЦена,ДатаПоследнегоПоступления,ОстатокНачалоНедели,temp_mean_week,precip_sum_week,temp_max_week,temp_min_week,НДС,days_off_in_week,sales_lag_2w
0,2017-04-10,00-00000030,Биогумус Флорист Бутон 120мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,62.0,шт,Штучный товар,Неизвестный поставщик,47.34,2017-04-02 12:00:00,10.0,6.142857,7.9,17.2,-5.3,0.18,0,0.0
14,2017-04-10,00-00000066,Энерген Аква 10мл. Стимулятор роста растений.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,41.0,шт,Штучный товар,Колхоз,31.41,2017-04-02 12:00:00,5.0,6.142857,7.9,17.2,-5.3,0.18,0,0.0
13,2017-04-10,00-00000058,Фосфатовит универсальный 220мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,125.0,шт,Штучный товар,Неизвестный поставщик,95.70,2017-04-02 12:00:00,5.0,6.142857,7.9,17.2,-5.3,0.18,0,0.0
12,2017-04-10,00-00000052,Удобрение для цитрусовых 285мл.,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,122.5,шт,Штучный товар,"ИП Алексеев Н.С, ПЕНЗА САДОВИТА",94.00,2017-04-02 12:00:00,2.0,6.142857,7.9,17.2,-5.3,0.18,0,0.0
11,2017-04-10,00-00000046,Партенокарпин-био 3мл. Стимулятор плодообразов...,01 Химия для сада и огорода,"01 Жидкие удобрения, стимуляторы, подкормки",0.0,83.0,шт,Штучный товар,Неизвестный поставщик,63.36,2017-04-02 12:00:00,3.0,6.142857,7.9,17.2,-5.3,0.18,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2496008,2025-10-27,РТ-00038161,М10 анкер латунный забивной (цанга) 2шт.,03 Стройка,19 Скобяные изделия ШТУЧНЫЕ,0.0,57.0,шт,Штучный товар,Неизвестный поставщик,24.51,2025-10-25 20:00:00,9.0,6.257143,18.3,11.5,1.8,0.20,0,0.0
2495999,2025-10-27,РТ-00038152,6х60 шуруп крючок Г-образный 2шт.,03 Стройка,19 Скобяные изделия ШТУЧНЫЕ,0.0,12.0,шт,Штучный товар,Неизвестный поставщик,5.12,2025-10-25 20:00:00,78.0,6.257143,18.3,11.5,1.8,0.20,0,0.0
2495987,2025-10-27,РТ-00038140,5х40 шуруп крючок О-образный 2шт.,03 Стройка,19 Скобяные изделия ШТУЧНЫЕ,0.0,11.0,шт,Штучный товар,Неизвестный поставщик,4.73,2025-10-25 20:00:00,100.0,6.257143,18.3,11.5,1.8,0.20,0,0.0
2495963,2025-10-27,РТ-00038116,105х105х90 уголок крепежный усиленный ОЦ,03 Стройка,19 Скобяные изделия ШТУЧНЫЕ,0.0,64.0,шт,Штучный товар,Неизвестный поставщик,27.83,2025-10-24 13:29:49,27.0,6.257143,18.3,11.5,1.8,0.20,0,0.0


In [373]:
# Сохраним итоговый df
agg_df.to_csv("features/features_one_week.csv", sep=';', index=False)